# sketch2stl · Run the app (GUI) in Colab

Draw a 2-D shape → it is recognised and cleaned up → extrude or revolve it, add or cut → download a printable **STL**.

The app downloads both trained models from the Hugging Face Hub on start-up:

| | model | what it does |
|---|---|---|
| ML1 | [pakiino/sketch2stl-recognizer](https://huggingface.co/pakiino/sketch2stl-recognizer) · trained from scratch | names every outline: line / arc / circle / rect / polyline |
| ML2 | [pakiino/sketch2stl-revolve-or-mirror](https://huggingface.co/pakiino/sketch2stl-revolve-or-mirror) · fine-tuned ResNet-18 | in *half + centreline* mode, decides **revolve** or **mirror-extrude** (you can override) |

**Runtime → Run all.** No GPU needed. Open the `https://….gradio.live` link printed at the bottom (the app also shows inside the notebook). It stays up while this cell runs; stop it with ■.

Code: [SunKaiwen1023/sketch2stl](https://github.com/SunKaiwen1023/sketch2stl)

## 1 · Get the app

In [ ]:
import os, sys, subprocess, time, importlib.util
t0 = time.time()
REPO_URL, BRANCH = "https://github.com/SunKaiwen1023/sketch2stl", "main"
if os.path.basename(os.getcwd()) == "sketch2stl":      # re-run after an earlier run: go back up
    os.chdir("..")
if os.path.exists("sketch2stl"):                        # always run the LATEST code, not an old copy
    subprocess.run(["git", "-C", "sketch2stl", "fetch", "-q", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", "sketch2stl", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, "sketch2stl"], check=True)
rev = subprocess.run(["git", "-C", "sketch2stl", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: {rev} ({time.time()-t0:.0f}s)")

# Install only what this Colab is missing, one package at a time, so a problem names itself.
NEED = {"numpy": "numpy", "shapely": "shapely>=2.0", "trimesh": "trimesh>=4.0", "manifold3d": "manifold3d",
        "gradio": "gradio>=6", "sklearn": "scikit-learn", "skimage": "scikit-image", "joblib": "joblib",
        "matplotlib": "matplotlib", "huggingface_hub": "huggingface_hub", "onnxruntime": "onnxruntime"}
missing = [spec for mod, spec in NEED.items() if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec("gradio"):
    import importlib.metadata as md
    if int(md.version("gradio").split(".")[0]) < 6: missing.append("gradio>=6")
print("installing:", missing or "nothing")
failed = []
for spec in missing:
    r = subprocess.run(["uv", "pip", "install", "--system", "-q", spec], capture_output=True, text=True) \
        if subprocess.run(["which", "uv"], capture_output=True).returncode == 0 else None
    if r is None or r.returncode != 0:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], capture_output=True, text=True)
    print(f"  {spec:22s} {'ok' if r.returncode == 0 else 'FAILED'}")
    if r.returncode != 0:
        failed.append(spec); print(r.stderr[-1500:])
os.chdir("sketch2stl"); sys.path.insert(0, os.getcwd())
print(f"done in {time.time()-t0:.0f}s" + (f" - FAILED: {failed} (send this output to PK/Claude)" if failed else ""))

## 2 · Launch  (models are fetched from Hugging Face here)

In [ ]:
import gradio as gr
gr.close_all()        # stop any app left running from an earlier run, so you get a fresh link
import app
app.launch(share=True, debug=True)

## Things to try
* **Several shapes at once:** draw a rectangle with two circles inside it → one plate with two holes.
* **Layers (left):** click a layer, change its depth / start z / add-cut / visibility on the right → **Apply**. ▲▼ reorder, Delete, Undo.
* **Make symmetric:** tick it before *Add* to straighten a nearly-symmetric outline.
* **Half + centreline:** draw half a vase against the centreline → ML2 chooses revolve; pick the other option to override.
* **Export STL** and open it in any slicer.